# Explore MEDS & Tokenized Sequences

Quick inspection of the extraction pipeline output:
- **MEDS sequences** (`sequences.parquet`) — raw event dicts with timestamps, codes, values
- **Tokenized sequences** (`all_tokens.parquet`) — integer token IDs + vocabulary

Run `protoecg-pipeline extract` and `protoecg-pipeline tokenize` first.

In [ ]:
from pathlib import Path
from collections import Counter

import polars as pl

# Show all rows/columns without truncation
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(120)
pl.Config.set_tbl_width_chars(200)

# ── Configure which run to inspect ──
# Set to None for the full run, or an integer for a test run (e.g. 1000, 10000)
N_PATIENTS = 10000

BASE_DIR = Path("./data/processed")
if N_PATIENTS is not None:
    BASE_DIR = BASE_DIR / f"n{N_PATIENTS}"

SEQ_PATH = BASE_DIR / "sequences.parquet"
TOKENIZED_DIR = BASE_DIR / "tokenized"

print(f"Run: {'full' if N_PATIENTS is None else f'n{N_PATIENTS}'}")
print(f"Base dir: {BASE_DIR}")
print(f"Sequences: {SEQ_PATH}  (exists={SEQ_PATH.exists()})")
print(f"Tokenized: {TOKENIZED_DIR}  (exists={TOKENIZED_DIR.exists()})")

Run: n10000
Base dir: ../data/processed/n10000
Sequences: ../data/processed/n10000/sequences.parquet  (exists=False)
Tokenized: ../data/processed/n10000/tokenized  (exists=False)


## 1. Load MEDS Sequences

In [6]:
# Read parquet directly with polars (fast — no Python dict conversion)
seq_df = pl.read_parquet(SEQ_PATH)

n_hosps = seq_df["hospitalization_id"].n_unique()
lengths = seq_df.group_by("hospitalization_id").len().sort("len")

print(f"Hospitalizations: {n_hosps}")
print(f"Total events: {len(seq_df):,}")
print(f"Events per hosp: min={lengths['len'].min()}, median={lengths['len'].median():.0f}, "
      f"max={lengths['len'].max()}, mean={lengths['len'].mean():.0f}")

FileNotFoundError: No such file or directory (os error 2): ../data/processed/n10000/sequences.parquet

### Event type breakdown

In [3]:
# Extract prefix from code column using polars (vectorized, no Python loop)
prefix_counts = (
    seq_df.with_columns(pl.col("code").str.split("//").list.first().alias("prefix"))
    .group_by("prefix")
    .len()
    .sort("len", descending=True)
)

for row in prefix_counts.iter_rows(named=True):
    print(f"  {row['prefix']:25s} {row['len']:>8,}")

NameError: name 'seq_df' is not defined

### ECG events summary

In [4]:
# ECG summary using polars (vectorized)
ecg_df = seq_df.filter(pl.col("code").str.starts_with("ECG//"))

ecg_hosps = ecg_df["hospitalization_id"].n_unique()
print(f"Hosps with ECG data: {ecg_hosps}/{n_hosps}")
print(f"Total ECG events: {len(ecg_df)}")

# ECG event types (Class, Prototype, Similarity)
ecg_types = (
    ecg_df.with_columns(pl.col("code").str.split("/").list.get(2).alias("ecg_type"))
    .group_by("ecg_type")
    .len()
    .sort("len", descending=True)
)
print("\nECG event types:")
for row in ecg_types.iter_rows(named=True):
    print(f"  {row['ecg_type']:25s} {row['len']:>6,}")

NameError: name 'seq_df' is not defined

### Label events summary

In [ ]:
# Label summary using polars (vectorized)
label_df = seq_df.filter(pl.col("code").str.starts_with("LABEL//"))

label_counts = label_df.group_by("code").len().sort("len", descending=True)
print(f"Distinct labels: {len(label_counts)}")
print(f"Total label events: {len(label_df)}")
print()
for row in label_counts.head(20).iter_rows(named=True):
    print(f"  {row['code']:50s} {row['len']:>5}")

### Inspect a single hospitalization

In [ ]:
# ── Toggle: which type of hospitalization to inspect ──
# "ecg"        → must have at least one ECG event
# "icu_vitals" → must be admitted to ICU (ADT//icu) AND have vitals data (VITAL//)
INSPECT_MODE = "icu_vitals"  # <-- change to "ecg" to switch

if INSPECT_MODE == "ecg":
    candidate_hids = ecg_df["hospitalization_id"].unique().sort()
    criteria = "has ECG events"
elif INSPECT_MODE == "icu_vitals":
    icu_hids = (
        seq_df.filter(pl.col("code") == "ADT//icu")
        ["hospitalization_id"].unique()
    )
    vitals_hids = (
        seq_df.filter(pl.col("code").str.starts_with("VITAL//"))
        ["hospitalization_id"].unique()
    )
    candidate_hids = (
        pl.DataFrame({"hospitalization_id": icu_hids})
        .join(
            pl.DataFrame({"hospitalization_id": vitals_hids}),
            on="hospitalization_id",
        )
        ["hospitalization_id"].sort()
    )
    criteria = "ICU admission + vitals"
else:
    raise ValueError(f"Unknown INSPECT_MODE: {INSPECT_MODE!r}  (use 'ecg' or 'icu_vitals')")

print(f"Mode: {INSPECT_MODE}  ({criteria})")
print(f"Matching hospitalizations: {len(candidate_hids)}")

sample_hid = candidate_hids[0] if len(candidate_hids) > 0 else seq_df["hospitalization_id"].unique().sort()[0]

sample_events = (
    seq_df.filter(pl.col("hospitalization_id") == sample_hid)
    .sort("event_idx")
    .drop("hospitalization_id")
)
print(f"Hospitalization: {sample_hid}  ({len(sample_events)} events)")
print(f"Time span: {sample_events['time'][0]} -> {sample_events['time'][-1]}")
print()
sample_events

In [ ]:
# ECG events for this hospitalization
ecg_sample = sample_events.filter(pl.col("code").str.starts_with("ECG//"))
if len(ecg_sample) > 0:
    print(f"ECG events in {sample_hid}: {len(ecg_sample)}")
    ecg_sample.select("time", "code", "value")
else:
    print(f"No ECG events in {sample_hid}")

## 2. Tokenized Sequences

Requires running `protoecg-pipeline tokenize` first. Skip this section if not yet tokenized.

In [ ]:
import json

tokens_path = TOKENIZED_DIR / "all_tokens.parquet"
vocab_path = TOKENIZED_DIR / "vocab" / "vocab.json"

if not tokens_path.exists():
    print(f"Tokenized data not found at {tokens_path}")
    print(f"Run: protoecg-pipeline tokenize" + (f" -n {N_PATIENTS}" if N_PATIENTS else ""))
else:
    tok_df = pl.read_parquet(tokens_path)
    print(f"Tokenized sequences: {len(tok_df)}")
    print(tok_df.select("hospitalization_id", "token_count").describe())

In [ ]:
if vocab_path.exists():
    vocab = json.loads(vocab_path.read_text())
    id_to_token = {v: k for k, v in vocab.items()}
    print(f"Vocabulary size: {len(vocab)}")

    # Prefix breakdown
    vocab_prefixes = Counter()
    for tok in vocab:
        vocab_prefixes[tok.split("//")[0] if "//" in tok else tok] += 1
    print("\nVocab token prefixes:")
    for prefix, count in vocab_prefixes.most_common():
        print(f"  {prefix:25s} {count:>5}")
else:
    print(f"Vocab not found at {vocab_path}")

Vocab not found at ../data/tokenized_test/vocab/vocab.json


### Decode a tokenized sequence

In [ ]:
if tokens_path.exists() and vocab_path.exists():
    # Pick the same hospitalization we inspected above
    row = tok_df.filter(pl.col("hospitalization_id") == sample_hid)
    if len(row) == 0:
        row = tok_df.head(1)
        sample_hid_tok = row["hospitalization_id"][0]
    else:
        sample_hid_tok = sample_hid

    token_ids = row["token_ids"][0].to_list()
    decoded = [id_to_token.get(tid, f"?{tid}") for tid in token_ids]

    # Load just this one hospitalization's events for side-by-side view
    hosp_events = (
        seq_df.filter(pl.col("hospitalization_id") == sample_hid_tok)
        .sort("event_idx")
        .to_dicts()
    )

    # Build side-by-side: walk tokens and MEDS events in parallel
    side_rows = []
    evt_idx = 0
    for tok_pos, (tid, tok_str) in enumerate(zip(token_ids, decoded)):
        meds_time = ""
        meds_code = ""
        meds_value = None

        if tok_str.startswith("Q_"):
            prev_evt = evt_idx - 1 if evt_idx > 0 else 0
            if prev_evt < len(hosp_events):
                meds_value = hosp_events[prev_evt].get("value")
        elif tok_str in ("[BOS]", "[EOS]", "[PAD]", "[UNK]", "[SEP]", "[CONT]"):
            pass
        else:
            if evt_idx < len(hosp_events):
                evt = hosp_events[evt_idx]
                meds_time = str(evt["time"])
                meds_code = evt["code"]
                if evt.get("value") is None and evt.get("value_cat"):
                    meds_value = evt["value_cat"]
                evt_idx += 1

        side_rows.append({
            "tok#": tok_pos,
            "tok_id": tid,
            "token": tok_str,
            "meds_time": meds_time,
            "meds_code": meds_code,
            "value": meds_value,
        })

    print(f"Hospitalization: {sample_hid_tok}")
    print(f"Tokens: {len(token_ids)}  |  MEDS events: {len(hosp_events)}")
    print()

    df_side = pl.DataFrame(side_rows)
    df_side.head(80)
else:
    print("Tokenized data or vocab not available.")